# Data preparation.

To get the data ready for modeling the following actions are performed:
- **Feature engineering**
- **Target creation**
- **Chronological split**
- **Scaling**
- **Sliding windows**


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import config
print(config.TRAIN_RATIO, config.VAL_RATIO, config.TEST_RATIO)

0.7 0.15 0.15


Next: [`03_modeling.ipynb`](03_modeling.ipynb)

In [3]:
print("LOOKBACK =", config.LOOKBACK)
print("USE_VOLUME =", config.USE_VOLUME)

LOOKBACK = 60
USE_VOLUME = False


In [5]:
import os
print(os.listdir(PROJECT_ROOT / "src"))

['features.py', 'plotting.py', 'data_loader.py']


In [6]:
import inspect
from src import data_loader

print(inspect.getsource(data_loader))

"""
data_loader.py - 

Handles the reading, cleaning, and validation of daily OHLCV stock data.

It is based on the CRISP-DM phase 2: Data Understanding findings.

Responsibilities
----------------
- Read the raw CSV.
- Validate the data (no missing values, positive prices, unique dates...).
- Optionally select a subset of the data from a ticker symbol.
- Return a clean DataFrame indexed by date, sorted from oldest to newest.

"""

from pathlib import Path
from typing import IO, List, Optional, Tuple, Union

import numpy as np
import pandas as pd

import config

# Columns every downstream step expects to find.
REQUIRED_COLUMNS = ["Open", "High", "Low", "Close", "Volume"]
PRICE_COLUMNS = ["Open", "High", "Low", "Close"]


def validate_stock_data(df: pd.DataFrame) -> List[str]:
    """
    Raise a ValueError if the data breaks basic assumptions.

    Returns a list of warnings for issues that are suspicious but not fatal,
    so the caller can decide whether to show them.
    """
    mis

In [7]:
print([name for name in dir(data_loader) if not name.startswith("_")])

['IO', 'List', 'Optional', 'PRICE_COLUMNS', 'Path', 'REQUIRED_COLUMNS', 'Tuple', 'Union', 'config', 'load_stock_data', 'np', 'pd', 'validate_stock_data']


In [8]:
from src import features

print([name for name in dir(features) if not name.startswith("_")])

['ENGINEERED_FEATURES', 'VOLUME_FEATURES', 'add_features', 'add_target', 'build_inference_frame', 'build_modeling_frame', 'config', 'get_feature_columns', 'np', 'pd']


In [9]:
print(inspect.getsource(features.build_modeling_frame))

def build_modeling_frame(df: pd.DataFrame, ticker: str = config.TICKER) -> pd.DataFrame:
    """
    Full feature-engineering pipeline for TRAINING/EVALUATION:
    features + target, with incomplete rows removed.

    Removes the first row (no previous close -> no return) and the last row
    (no next day -> no target).
    """
    out = add_target(add_features(df), ticker)
    return out.dropna(subset=ENGINEERED_FEATURES + ["target"])



In [10]:
print(inspect.getsource(features.build_modeling_frame))

def build_modeling_frame(df: pd.DataFrame, ticker: str = config.TICKER) -> pd.DataFrame:
    """
    Full feature-engineering pipeline for TRAINING/EVALUATION:
    features + target, with incomplete rows removed.

    Removes the first row (no previous close -> no return) and the last row
    (no next day -> no target).
    """
    out = add_target(add_features(df), ticker)
    return out.dropna(subset=ENGINEERED_FEATURES + ["target"])



In [11]:
print(inspect.getsource(features.add_target))

def add_target(df: pd.DataFrame, ticker: str = config.TICKER) -> pd.DataFrame:
    """
    Add the prediction target for each row t, which describes day t+1.

    shift(-1) moves every value one row UP, so row t receives tomorrow's value.
    The last row gets NaN because its "tomorrow" is not in the data yet -
    that is exactly the day we will predict in deployment.

    Columns added
    -------------
    target      : what the network learns (next log return).
    next_close  : tomorrow's actual close (used to evaluate in dollars).
    target_date : tomorrow's date (used for plots).
    """
    out = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        out["next_close", ticker] = out["Close", ticker].shift(-1)
        out["target_date"] = out.index.to_series().shift(-1)
        out["target", ticker] = out["log_return", ticker]
    else:
        out["next_close"] = out["Close"].shift(-1)
        out["target_date"] = out.index.to_series().shift(-1)
        out["target"] =

In [12]:
print(inspect.getsource(features.get_feature_columns))

def get_feature_columns(use_volume: bool) -> list:
    """
    Return the list of input columns.

    use_volume=False -> ["log_return"]                                  (price only)
    use_volume=True  -> ["log_return", "log_volume", "volume_change"]   (price + volume)
    """
    if use_volume:
        columns += VOLUME_FEATURES
    return columns



In [14]:
import os

for root, dirs, files in os.walk(PROJECT_ROOT):
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)
            text = open(path, "r", errors="ignore").read()
            if (
                "TRAIN_RATIO" in text
                or "StandardScaler" in text
                or "MinMaxScaler" in text
            ):
                print(path)

/opt/aima-python/Stock_Price_DL/config.py
/opt/aima-python/Stock_Price_DL/.ipynb_checkpoints/config-checkpoint.py


In [16]:
for root, dirs, files in os.walk(PROJECT_ROOT):
    for file in files:
        if file.endswith(".ipynb"):
            path = os.path.join(root, file)
            text = open(path, "r", errors="ignore").read()
            if (
                "TRAIN_RATIO" in text
                or "StandardScaler" in text
                or "MinMaxScaler" in text
                or "RobustScaler" in text
            ):
                print(path)

/opt/aima-python/Stock_Price_DL/notebooks/02_data_preparation.ipynb


In [18]:
import json

p = PROJECT_ROOT / "notebooks" / "03_modeling.ipynb"
nb = json.load(open(p))

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] == "code":
        src = "".join(cell["source"])
        if any(word in src for word in [
            "TRAIN_RATIO", "VAL_RATIO", "TEST_RATIO",
            "fit_transform", "transform(", "scaler",
            "train", "validation", "test"
        ]):
            print(f"\n--- CELL {i} ---")
            print(src)

In [19]:
for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] == "code":
        print(f"\n--- CELL {i} ---")
        print("".join(cell["source"]))



--- CELL 1 ---


--- CELL 3 ---



In [20]:
p = PROJECT_ROOT / "notebooks" / "02_data_preparation.ipynb"
nb2 = json.load(open(p))

for i, cell in enumerate(nb2["cells"]):
    if cell["cell_type"] == "code":
        print(f"\n--- CELL {i} ---")
        print("".join(cell["source"]))



--- CELL 1 ---
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import config
print(config.TRAIN_RATIO, config.VAL_RATIO, config.TEST_RATIO)

--- CELL 3 ---
print("LOOKBACK =", config.LOOKBACK)
print("USE_VOLUME =", config.USE_VOLUME)

--- CELL 4 ---
import inspect
from src import data_prep

print(inspect.getsource(data_prep))


--- CELL 5 ---
import os
print(os.listdir(PROJECT_ROOT / "src"))

--- CELL 6 ---
import inspect
from src import data_loader

print(inspect.getsource(data_loader))

--- CELL 7 ---
print([name for name in dir(data_loader) if not name.startswith("_")])

--- CELL 8 ---
from src import features

print([name for name in dir(features) if not name.startswith("_")])

--- CELL 9 ---
print(inspect.getsource(features.build_modeling_frame))

--- CELL 10 ---
print(inspect.getsource(features.build_modeling_frame))

--- CELL 11 ---
print(inspect.getsource(features.add_target))

--- CELL 12 ---
print(inspect.getsource

In [21]:
import subprocess

original = subprocess.check_output(
    ["git", "show", "HEAD:notebooks/02_data_preparation.ipynb"],
    cwd=PROJECT_ROOT
)

nb_original = json.loads(original)

for i, cell in enumerate(nb_original["cells"]):
    if cell["cell_type"] == "code":
        print(f"\n--- ORIGINAL CELL {i} ---")
        print("".join(cell["source"]))


--- ORIGINAL CELL 1 ---


--- ORIGINAL CELL 3 ---



In [23]:
import subprocess

print(
    subprocess.check_output(
        ["git", "ls-tree", "-r", "--name-only", "HEAD"],
        cwd=PROJECT_ROOT,
        text=True
    )
)

.gitattributes
.github/skills/git-workflow/SKILL.md
.gitignore
.ipynb_checkpoints/README-checkpoint.md
AGENTS.md
README.md
config.py
docs/01_crisp_dm_workflow.md
docs/git-workflow.md
notebooks/.ipynb_checkpoints/01_business_and_data_understanding-checkpoint.ipynb
notebooks/.ipynb_checkpoints/02_data_preparation-checkpoint.ipynb
notebooks/.ipynb_checkpoints/03_modeling-checkpoint.ipynb
notebooks/.ipynb_checkpoints/04_evaluation-checkpoint.ipynb
notebooks/.ipynb_checkpoints/05_deployment-checkpoint.ipynb
notebooks/01_business_and_data_understanding.ipynb
notebooks/02_data_preparation.ipynb
notebooks/03_modeling.ipynb
notebooks/04_evaluation.ipynb
notebooks/05_deployment.ipynb
src/data_loader.py
src/features.py
src/plotting.py



In [24]:
print(inspect.getsource(features.add_features))


def add_features(df: pd.DataFrame, ticker: str = config.TICKER) -> pd.DataFrame:
    """
    Add engineered feature columns.

    ticker: used for selecting the MultiIndex column.

    Columns added
    -------------
    log_return    : ln(Close_t / Close_{t-1})
    log_volume    : ln(1 + Volume_t)
    volume_change : log_volume_t - log_volume_{t-1}
    """
    out = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        out["log_return", ticker] = np.log(out["Close", ticker] / out["Close", ticker].shift(1))
        out["log_volume", ticker] = np.log1p(out["Volume", ticker])
        out["volume_change", ticker] = out["log_volume", ticker].diff()
    else:
        out["log_return"] = np.log(out["Close"] / out["Close"].shift(1))
        out["log_volume"] = np.log1p(out["Volume"])
        out["volume_change"] = out["log_volume"].diff()
    return out



In [25]:
print(inspect.getsource(features.get_feature_columns))


def get_feature_columns(use_volume: bool) -> list:
    """
    Return the list of input columns.

    use_volume=False -> ["log_return"]                                  (price only)
    use_volume=True  -> ["log_return", "log_volume", "volume_change"]   (price + volume)
    """
    if use_volume:
        columns += VOLUME_FEATURES
    return columns



In [27]:
import os

for root, dirs, files in os.walk(PROJECT_ROOT / "src"):
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)
            text = open(path, "r", errors="ignore").read()
            if (
                "split" in text.lower()
                or "scaler" in text.lower()
                or "window" in text.lower()
                or "sequence" in text.lower()
            ):
                print(path)

/opt/aima-python/Stock_Price_DL/src/plotting.py


In [28]:
path = PROJECT_ROOT / "src" / "plotting.py"

for line_number, line in enumerate(open(path), start=1):
    if any(word in line.lower() for word in ["split", "scaler",
"window", "sequence"]):
        print(line_number, line.strip())

43 SPLIT_COLORS = {"train": SERIES[0], "val": SERIES[1], "test": SERIES[2]}


In [29]:
print(inspect.getsource(config))


"""
config.py - Central configuration for the whole project.

"""

from pathlib import Path

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parent

DATA_PATH = PROJECT_ROOT / "data" / "SnP_daily_update.csv"   # raw input data
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"                   # trained models, scalers, metadata
PLOTS_DIR = ARTIFACTS_DIR / "plots"                          # figures saved by the evaluation step
METRICS_PATH = ARTIFACTS_DIR / "metrics.csv"                 # model comparison table

TICKER = "AAPL"  # only used for titles/labels

# ---------------------------------------------------------------------------
# Data selection
# ---------------------------------------------------------------------------
# The file starts in 1/4/2010.
# Set to None to use the full history.
START_DATE = "2010-01-04"

# -------------

In [30]:
import subprocess

print(
    subprocess.check_output(
        ["git", "branch", "--show-current"],
        cwd=PROJECT_ROOT,
        text=True
    )
)

main



In [31]:
print(
    subprocess.check_output(
        ["git", "switch", "-c", "basma-model2"],
        cwd=PROJECT_ROOT,
        text=True,
        stderr=subprocess.STDOUT
    )
)

Switched to a new branch 'basma-model2'



In [32]:
print(
    subprocess.check_output(
        ["git", "branch", "--show-current"],
        cwd=PROJECT_ROOT,
        text=True
    )
)


basma-model2



In [34]:
print(inspect.signature(data_loader.load_stock_data))
print(inspect.signature(features.build_modeling_frame))


(source: Union[str, pathlib.Path, IO] = PosixPath('/opt/aima-python/Stock_Price_DL/data/SnP_daily_update.csv'), ticker: Optional[str] = 'AAPL', start_date: Optional[str] = '2010-01-04', end_date: Optional[str] = None, verbose: bool = True) -> pandas.core.frame.DataFrame
(df: pandas.core.frame.DataFrame, ticker: str = 'AAPL') -> pandas.core.frame.DataFrame


In [37]:
import os

for root, dirs, files in os.walk("/opt"):
    if "SnP_daily_update.csv" in files:
        print(os.path.join(root, "SnP_daily_update.csv"))

In [39]:
import pandas as pd

check = pd.read_csv(config.DATA_PATH, header=None, nrows=5)
print(check.iloc[:, :8])

            0                   1                   2      3      4  \
0       Price               Close               Close  Close  Close   
1      Ticker                   A                AAPL   ABBV   ABNB   
2        Date                 NaN                 NaN    NaN    NaN   
3  2010-01-04  19.772951126098633   6.400960922241211    NaN    NaN   
4  2010-01-05   19.55816650390625  6.4120259284973145    NaN    NaN   

                    5                   6                   7  
0               Close               Close               Close  
1                 ABT                ACGL                 ACN  
2                 NaN                 NaN                 NaN  
3  18.078800201416016    7.60190486907959  30.856338500976562  
4  17.932741165161133  7.5765485763549805  31.047033309936523  


In [40]:
raw_data = pd.read_csv(
    config.DATA_PATH,
    header=[0, 1],
    skiprows=[2],
    index_col=0,
    parse_dates=True
)

print(raw_data.shape)
print(raw_data.head())

(4210, 2515)
Price           Close                                                      \
Ticker              A      AAPL ABBV ABNB        ABT      ACGL        ACN   
2010-01-04  19.772951  6.400961  NaN  NaN  18.078800  7.601905  30.856339   
2010-01-05  19.558167  6.412026  NaN  NaN  17.932741  7.576549  31.047033   
2010-01-06  19.488672  6.310036  NaN  NaN  18.032326  7.543795  31.377081   
2010-01-07  19.463404  6.298370  NaN  NaN  18.181715  7.499420  31.347759   
2010-01-08  19.457090  6.340245  NaN  NaN  18.274658  7.484628  31.223053   

Price                                        ...   Volume                    \
Ticker           ADBE        ADI        ADM  ...       WY     WYNN      XEL   
2010-01-04  37.090000  21.703712  20.029284  ...  1832400  4741400  2670400   
2010-01-05  37.700001  21.669453  20.137486  ...  1724500  5644300  4321400   
2010-01-06  37.619999  21.628340  20.086567  ...  2254400  2738800  2164500   
2010-01-07  36.889999  21.457008  19.876535  ...  14

In [42]:
aapl_data = raw_data.xs(config.TICKER, axis=1, level=1)

print(aapl_data.shape)
print(aapl_data.columns)
print(aapl_data.head())

(4210, 5)
Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')
Price          Close      High       Low      Open     Volume
2010-01-04  6.400961  6.415617  6.352208  6.383613  493729600
2010-01-05  6.412026  6.448217  6.378228  6.418606  601904800
2010-01-06  6.310036  6.437451  6.303456  6.412028  552160000
2010-01-07  6.298370  6.340842  6.252608  6.333365  477131200
2010-01-08  6.340245  6.340843  6.252909  6.289997  447610800


In [43]:
data = features.build_modeling_frame(aapl_data)

print(data.shape)
print(data.head())

(4209, 11)
Price          Close      High       Low      Open     Volume  log_return  \
2010-01-05  6.412026  6.448217  6.378228  6.418606  601904800    0.001727   
2010-01-06  6.310036  6.437451  6.303456  6.412028  552160000   -0.016034   
2010-01-07  6.298370  6.340842  6.252608  6.333365  477131200   -0.001851   
2010-01-08  6.340245  6.340843  6.252909  6.289997  447610800    0.006627   
2010-01-11  6.284315  6.370753  6.234664  6.364771  462229600   -0.008861   

Price       log_volume  volume_change  next_close target_date    target  
2010-01-05   20.215610       0.198111    6.310036  2010-01-06  0.001727  
2010-01-06   20.129348      -0.086261    6.298370  2010-01-07 -0.016034  
2010-01-07   19.983302      -0.146046    6.340245  2010-01-08 -0.001851  
2010-01-08   19.919435      -0.063867    6.284315  2010-01-11  0.006627  
2010-01-11   19.951572       0.032138    6.212829  2010-01-12 -0.008861  


In [44]:
n = len(data)

train_end = int(n * config.TRAIN_RATIO)
val_end = train_end + int(n * config.VAL_RATIO)

train_data = data.iloc[:train_end].copy()
val_data = data.iloc[train_end:val_end].copy()
test_data = data.iloc[val_end:].copy()

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (2946, 11)
Validation: (631, 11)
Test: (632, 11)


In [46]:
feature_cols = ["log_return", "log_volume", "volume_change"]

print(feature_cols)


['log_return', 'log_volume', 'volume_change']


In [47]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_scaled = train_data.copy()
val_scaled = val_data.copy()
test_scaled = test_data.copy()

train_scaled[feature_cols] = scaler.fit_transform(train_data[feature_cols])
val_scaled[feature_cols] = scaler.transform(val_data[feature_cols])
test_scaled[feature_cols] = scaler.transform(test_data[feature_cols])

print(feature_cols)
print(train_scaled[feature_cols].head())

['log_return', 'log_volume', 'volume_change']
Price       log_return  log_volume  volume_change
2010-01-05    0.037632    1.448488       0.619346
2010-01-06   -0.963380    1.331094      -0.266666
2010-01-07   -0.164008    1.132339      -0.452936
2010-01-08    0.313760    1.045421      -0.196894
2010-01-11   -0.559094    1.089158       0.102226


In [48]:
import numpy as np

lookback = config.LOOKBACK

def make_windows(df, feature_cols, lookback):
    X = []
    y = []

    for i in range(lookback, len(df)):
        X.append(df[feature_cols].iloc[i-lookback:i].values)
        y.append(df["target"].iloc[i])

    return np.array(X), np.array(y)

X_train, y_train = make_windows(train_scaled, feature_cols, lookback)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (2886, 60, 3)
y_train shape: (2886,)


In [49]:
X_val, y_val = make_windows(val_scaled, feature_cols, lookback)
X_test, y_test = make_windows(test_scaled, feature_cols, lookback)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_val shape: (571, 60, 3)
y_val shape: (571,)
X_test shape: (572, 60, 3)
y_test shape: (572,)


In [50]:
print("Train:", train_data.index.min(), "to", train_data.index.max())
print("Validation:", val_data.index.min(), "to", val_data.index.max())
print("Test:", test_data.index.min(), "to", test_data.index.max())

Train: 2010-01-05 00:00:00 to 2021-09-16 00:00:00
Validation: 2021-09-17 00:00:00 to 2024-03-21 00:00:00
Test: 2024-03-22 00:00:00 to 2026-09-29 00:00:00
